# 第九课：同步、异步与并发基础

本课目标：

- 理解同步等待为什么浪费时间；
- 看懂`async def`、协程和`await`；
- 使用`asyncio.sleep()`模拟网络等待；
- 使用`asyncio.gather()`并发处理多个请求；
- 理解异步适合I/O等待，不会自动加速CPU计算；
- 使用pytest测试异步函数。

以后同时请求多个模型、下载多个网页或访问数据库时会用到这些概念。

## 1. 什么是同步执行

同步执行表示一件事完成后，才开始下一件事：

```text
请求A：等待1秒
请求B：等待1秒
请求C：等待1秒
总时间约3秒
```

下面使用`time.sleep()`模拟阻塞等待。

In [1]:
import time


def sync_request(name: str, delay: float) -> str:
    print(name, "开始")
    time.sleep(delay)
    print(name, "结束")
    return name


start = time.perf_counter()
results = [
    sync_request("A", 0.3),
    sync_request("B", 0.3),
    sync_request("C", 0.3),
]
seconds = time.perf_counter() - start

print(results)
print(f"用时：{seconds:.2f}秒")

A 开始
A 结束
B 开始
B 结束
C 开始
C 结束
['A', 'B', 'C']
用时：0.90秒


## 2. `async def`和协程

使用`async def`定义异步函数：

```python
async def async_request():
    ...
```

调用普通函数会立即执行函数体；调用异步函数首先得到一个**协程对象**，它描述一项可以被异步调度的工作。

In [2]:
import asyncio


async def say_hello() -> str:
    return "hello"


coroutine = say_hello()
print(coroutine)
print(type(coroutine))

# 关闭这个尚未运行的协程，避免Notebook产生警告。
coroutine.close()

<coroutine object say_hello at 0x000001B6BE78D430>
<class 'coroutine'>


## 3. `await`是什么意思

`await`表示等待一个异步操作完成。在等待网络、磁盘或定时器时，事件循环可以先运行其他任务。

`await`通常只能写在`async def`内部。Jupyter为了方便，也允许在代码单元格顶层直接使用`await`。

In [3]:
async def async_request(name: str, delay: float) -> str:
    print(name, "开始")
    await asyncio.sleep(delay)
    print(name, "结束")
    return name


result = await async_request("A", 0.3)
print(result)

A 开始
A 结束
A


### `time.sleep()`和`asyncio.sleep()`的区别

```python
time.sleep(1)
```

会阻塞当前线程，事件循环不能利用这段等待时间。

```python
await asyncio.sleep(1)
```

会暂停当前协程并把执行机会交还事件循环，让其他任务先运行。

异步函数中不要随意使用`time.sleep()`。

## 4. 写了`async`不代表自动并发

下面虽然使用异步函数，但每次都立刻`await`，所以仍然按顺序执行：

In [ ]:
start = time.perf_counter()

sequential_results = []
for name in ["A", "B", "C"]:
    result = await async_request(name, 0.3)
    sequential_results.append(result)

seconds = time.perf_counter() - start
print(sequential_results)
print(f"顺序异步用时：{seconds:.2f}秒")

## 5. 使用`asyncio.gather()`并发等待

先创建多项协程工作，再把它们一起交给`gather()`：

In [ ]:
start = time.perf_counter()

coroutines = [
    async_request("A", 0.3),
    async_request("B", 0.3),
    async_request("C", 0.3),
]
concurrent_results = await asyncio.gather(*coroutines)

seconds = time.perf_counter() - start
print(concurrent_results)
print(f"并发用时：{seconds:.2f}秒")

`*coroutines`表示把列表中的多个协程作为独立参数传给`gather()`：

```python
asyncio.gather(*coroutines)
```

近似于：

```python
asyncio.gather(coroutine_a, coroutine_b, coroutine_c)
```

`gather()`返回的结果顺序通常与传入协程的顺序一致，不取决于谁先完成。

## 6. 异步适合什么任务

适合大量等待外部响应的I/O任务：

- 调用大模型API；
- 下载网页；
- 访问数据库；
- 读写网络连接。

异步不会自动加速纯CPU计算，例如大型矩阵运算、图片编码或模型推理。CPU密集任务通常需要多进程、GPU或专门的任务系统。

## 7. 普通Python文件怎样启动异步程序

Notebook可以直接写`await`，普通`.py`文件通常使用：

```python
async def main():
    result = await async_request("A", 0.1)
    print(result)


if __name__ == "__main__":
    asyncio.run(main())
```

`asyncio.run()`创建事件循环、运行协程，并在完成后关闭事件循环。

## 8. 使用pytest测试异步函数

本课不额外安装异步测试插件，而是在普通测试中使用`asyncio.run()`：

```python
def test_async_function():
    result = asyncio.run(mock_llm_call("你好", delay=0))
    assert result == "回答：你好"
```

这样仍然是普通pytest测试函数。

# 编程细讲：异步代码是怎样写出来的

前面解释了概念，下面专门学习代码写法。我们先使用与作业不同的`download()`例子，理解后再把结构迁移到`mock_llm_call()`。

先记住三个层次：

```text
async def       定义一项异步工作
await           在异步代码中等待一项工作
asyncio.run()   从普通Python程序启动最外层异步工作
```

## A. 普通函数调用与异步函数调用

普通函数调用时，函数体立即执行并返回最终结果：

```python
def normal():
    return "完成"

result = normal()  # result直接是"完成"
```

异步函数调用时，先产生协程对象：

```python
async def asynchronous():
    return "完成"

work = asynchronous()  # work是协程对象，不是"完成"
result = await work    # 真正运行并等待结果
```

In [4]:
def normal_function() -> str:
    print("普通函数体执行")
    return "普通结果"


async def async_function() -> str:
    print("异步函数体执行")
    return "异步结果"


normal_result = normal_function()
print("普通调用得到：", normal_result)

async_work = async_function()
print("异步调用暂时得到：", async_work)

async_result = await async_work
print("await以后得到：", async_result)

普通函数体执行
普通调用得到： 普通结果
异步调用暂时得到： <coroutine object async_function at 0x000001B6BFDAC7C0>
异步函数体执行
await以后得到： 异步结果


## B. 事件循环是什么

事件循环可以理解为一个任务调度员：

```text
运行任务A
任务A遇到await，需要等待网络
暂时放下A，运行任务B
任务B也遇到await，运行任务C
A的网络结果到了，继续运行A
```

`await`不是让整个程序什么都不做，而是告诉事件循环：当前协程需要等待，你可以先运行其他已经准备好的任务。

Jupyter已经运行着事件循环，所以可以在单元格中直接写`await`。普通`.py`文件通常用`asyncio.run()`创建并启动事件循环。

## C. 一步步写一个异步函数

目标：模拟下载一个文件，等待结束后返回文件名。

第一步，函数需要异步等待，所以使用`async def`：

```python
async def download(name: str, delay: float) -> str:
```

第二步，用异步睡眠模拟网络等待：

```python
    await asyncio.sleep(delay)
```

第三步，等待完成后返回普通字符串：

```python
    return f"已下载：{name}"
```

合起来如下。

In [5]:
async def download(name: str, delay: float) -> str:
    print(name, "开始等待")
    await asyncio.sleep(delay)
    print(name, "等待结束")
    return f"已下载：{name}"


result = await download("a.pdf", 0.2)
print("最终返回值：", result)

a.pdf 开始等待
a.pdf 等待结束
最终返回值： 已下载：a.pdf


## D. 一步步写顺序异步函数

如果有多个文件，可以先写最容易理解的顺序版本：

```python
async def download_sequential(names: list[str]) -> list[str]:
    results: list[str] = []

    for name in names:
        result = await download(name, 0.1)
        results.append(result)

    return results
```

逐行理解：

1. 创建空结果列表；
2. 取出第一个名字；
3. `await`第一个下载完全结束；
4. 保存结果；
5. 才开始下一个名字。

它是异步代码，但不是并发执行。

In [6]:
async def download_sequential(names: list[str]) -> list[str]:
    results: list[str] = []
    for name in names:
        result = await download(name, 0.1)
        results.append(result)
    return results


start = time.perf_counter()
results = await download_sequential(["a.pdf", "b.pdf", "c.pdf"])
print(results)
print(f"顺序用时：{time.perf_counter() - start:.2f}秒")

a.pdf 开始等待
a.pdf 等待结束
b.pdf 开始等待
b.pdf 等待结束
c.pdf 开始等待
c.pdf 等待结束
['已下载：a.pdf', '已下载：b.pdf', '已下载：c.pdf']
顺序用时：0.33秒


## E. 一步步改成并发版本

顺序版本在循环中立刻`await`，所以一次只等待一个任务。并发版本要先准备所有协程：

```python
coroutines = []
for name in names:
    work = download(name, 0.1)
    coroutines.append(work)
```

此时列表中是协程对象，它们描述要做的工作。然后统一交给`gather()`：

```python
results = await asyncio.gather(*coroutines)
```

`gather()`会调度这些协程，并等待它们全部完成。

In [7]:
async def download_concurrent(names: list[str]) -> list[str]:
    coroutines = []
    for name in names:
        work = download(name, 0.1)
        coroutines.append(work)

    results = await asyncio.gather(*coroutines)
    return list(results)


start = time.perf_counter()
results = await download_concurrent(["a.pdf", "b.pdf", "c.pdf"])
print(results)
print(f"并发用时：{time.perf_counter() - start:.2f}秒")

a.pdf 开始等待
b.pdf 开始等待
c.pdf 开始等待
a.pdf 等待结束
b.pdf 等待结束
c.pdf 等待结束
['已下载：a.pdf', '已下载：b.pdf', '已下载：c.pdf']
并发用时：0.11秒


## F. `*coroutines`为什么有一个星号

先用普通函数理解星号拆包：

In [8]:
def show_three(first, second, third):
    print(first, second, third)


values = ["A", "B", "C"]
show_three(*values)

# 上面等价于：
show_three("A", "B", "C")

A B C
A B C


同理：

```python
await asyncio.gather(*coroutines)
```

会把协程列表拆成多个独立参数：

```python
await asyncio.gather(coroutine_a, coroutine_b, coroutine_c)
```

星号不是异步专用语法，它是Python的序列拆包语法。

## G. 初学异步最常见的错误

### 1. 忘记`await`

```python
result = download("a.pdf", 0.1)
```

`result`是协程对象，不是下载结果。

### 2. 在异步函数中使用`time.sleep()`

它会阻塞事件循环，使其他协程也无法继续。

### 3. 以为`async def`自动并发

循环中每次立刻`await`仍然是顺序执行。必须把多项工作一起交给`gather()`或创建任务。

### 4. 在Jupyter中调用`asyncio.run()`

Jupyter已经有事件循环，单元格中通常直接使用`await`。`asyncio.run()`主要用于普通`.py`文件的最外层入口。

## H. 把示例迁移到本课作业

不要一次写完三个函数，按下面顺序进行：

### 第一步：`mock_llm_call()`

它与刚才的`download()`结构相同：验证参数 → 异步等待 → 返回字符串。

完成后只运行：

```powershell
python -m pytest -v -k mock_llm_call
```

### 第二步：`run_sequential()`

参考`download_sequential()`：创建结果列表 → 循环 → 每次`await` → 添加结果。

### 第三步：`run_concurrent()`

参考`download_concurrent()`：先创建协程列表 → `await asyncio.gather(*coroutines)` → 返回结果。

每完成一步就运行测试，不要等全部写完再检查。

## 9. 动手作业

文件：

- `async_service.py`：完成三个异步函数；
- `test_async_service.py`：已有测试，并需要补两个测试。

步骤：

1. 运行`python -m pytest -v`观察初始失败；
2. 完成`mock_llm_call()`；
3. 完成`run_sequential()`；
4. 完成`run_concurrent()`；
5. 补充测试文件末尾的两个测试；
6. 反复运行直到全部通过。

In [ ]:
import pytest

exit_code = pytest.main(["-v", "test_async_service.py"])
print("pytest退出码：", exit_code)
assert exit_code == pytest.ExitCode.OK

## 10. 理解题

1. 调用`async def`函数时，为什么不会像普通函数一样立即得到最终返回值？
- 因为他创建的是一个协程对象
2. `await`的作用是什么？
- 等待这个异步任务完成，取得它的返回值；等待期间允许事件循环执行其他任务。
3. 为什么在异步函数中通常不应该使用`time.sleep()`？
- `time.sleep()`会阻塞当前线程和事件循环，导致其他协程也无法运行；`await asyncio.sleep()`只暂停当前协程，事件循环可以继续执行其他协程。
4. 仅仅把函数写成`async def`，多个调用就会自动并发吗？为什么？
- 不会
- 你还需要用asyncio.gather来使多个调用并发执行
5. `asyncio.gather()`解决了什么问题？返回结果顺序如何确定？
- 让多个调用并发执行,返回顺序和输入的顺序一致
6. 为什么异步适合调用大模型API，但不会自动加速CPU密集型模型推理？
- API 调用主要在等待网络和远程服务器，属于 I/O 密集型任务。等待期间，事件循环可以执行其他协程，因此适合异步并发。
- 模型推理主要在执行矩阵运算，属于计算密集型任务，没有可供异步利用的等待时间，因此异步不会自动增加计算速度。